# ⚡ ClipForge Cloud: 100% Free AI Short-Form Clipping Studio

Welcome to **ClipForge Cloud**! This notebook runs the entire ClipForge platform on Google's free cloud servers with an **NVIDIA GPU** and **13 GB of RAM**.

> [!TIP]
> **You can close your laptop!** This runs on Google's datacenter. All generated clips and your database will be permanently saved to your Google Drive.

### Quick Start (Takes 1 Minute):
1. **Runtime Type:** Ensure GPU is enabled (`Runtime` -> `Change runtime type` -> select `T4 GPU`).
2. **Run All:** Click `Runtime` -> `Run all` (or press `Ctrl + F9`).
3. **Click the Public Link:** A live `trycloudflare.com` URL will appear at the bottom. Click it to open your studio!

### Step 1: Mount Google Drive (Permanent Storage for Clips & Database)

In [ ]:
import os
from google.colab import drive

# Mount your personal Google Drive
drive.mount('/content/drive')

# Create permanent storage folders in Google Drive
drive_base = '/content/drive/MyDrive/ClipForge'
drive_output = f'{drive_base}/output'
drive_uploads = f'{drive_base}/uploads'

os.makedirs(drive_output, exist_ok=True)
os.makedirs(drive_uploads, exist_ok=True)
print("✅ Google Drive successfully connected! Your clips will be saved in 'My Drive/ClipForge'")

### Step 2: Download Code & Install Dependencies (FFmpeg + Whisper)

In [ ]:
# Clone or pull latest ClipForge from GitHub
%cd /content
!rm -rf clipforge
!git clone https://github.com/imoniosarogodwin16-cmd/clipforge.git
%cd /content/clipforge

# Install FFmpeg and system tools
!apt-get update -qq && apt-get install -y -qq ffmpeg wget

# Install Python packages
!pip install -q -r requirements.txt

# Link outputs and database to Google Drive for permanent preservation
!rm -rf /content/clipforge/output /content/clipforge/uploads
os.symlink(drive_output, '/content/clipforge/output')
os.symlink(drive_uploads, '/content/clipforge/uploads')

# Copy existing clipforge.db to Drive if not present, and symlink it
drive_db = f'{drive_base}/clipforge.db'
if os.path.exists('/content/clipforge/clipforge.db') and not os.path.exists(drive_db):
    !cp /content/clipforge/clipforge.db {drive_db}
if os.path.exists(drive_db):
    !rm -f /content/clipforge/clipforge.db
    os.symlink(drive_db, '/content/clipforge/clipforge.db')

print("✅ Setup completed! FFmpeg and AI models ready on GPU.")

### Step 3: Launch ClipForge with Free Public HTTPS Cloud Link

In [ ]:
import subprocess
import time
import re

# Download standalone Cloudflare binary (fast & reliable)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Start ClipForge FastAPI server in background
server_proc = subprocess.Popen([
    "python", "-m", "uvicorn", "app:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

time.sleep(3)

# Start Cloudflare quick tunnel to get public HTTPS link
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "=" * 70)
if public_url:
    print("🚀 CLIPFORGE CLOUD IS LIVE!")
    print(f"👉 Open this link in your phone or computer:\n{public_url}")
    print("=" * 70 + "\n")
    print("💡 You can now close your laptop or use this link anywhere in the world.")
else:
    print("Tunnel took longer than usual. Check tunnel logs:")
    print(tunnel_proc.stderr.read()[:500])

# Keep cell alive to maintain background process
try:
    server_proc.wait()
except KeyboardInterrupt:
    print("Server stopped.")